# Notebook 00 — Data Loading & Structural Inspection

## Macro-Aviation-Tourism Demand Modelling and Forecasting for Poland

**Author:** Diego Marrero Ferrera  
**Repository:** [GitHub — Macro-Aviation-Tourism-Modeling](https://github.com/DieGodMF4/Macro-Aviation-Tourism-Modeling)  
**Last updated:** 2026-03

---

### Purpose

This notebook performs a **purely structural** inspection of every raw dataset used in the project. Its goal is to verify that all files exist, load correctly, and contain the expected columns, filter values, and time coverage — **before any analysis begins**.

Specifically, this notebook:

1. Catalogues all raw CSV files and confirms their presence on disk.
2. Loads each dataset and reports its shape, column names, data types, and memory footprint.
3. Enumerates the unique values of every categorical filter column (countries, units, indicators).
4. Checks temporal coverage (start date, end date, number of periods).
5. Verifies that the countries and variables required by the thesis are present.
6. Produces a consolidated **Data Inventory** and a **Filter Reference Table** for use in all subsequent notebooks (01–08).

No plots, no summary statistics, and no analysis are performed here — those belong in Notebook 01 (Descriptive EDA).

---

## 0. Environment Setup

In [59]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

# Display options for readable inspection output
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 60)

# --- PATH CONFIGURATION ---
# This notebook lives in notebooks/; data is in data/raw/ (one level up).
DATA_RAW = os.path.join('..', 'data', 'raw')

assert os.path.isdir(DATA_RAW), (
    f"Data directory not found: {os.path.abspath(DATA_RAW)}. "
    f"Make sure you are running this notebook from the notebooks/ folder."
)
print(f"Data directory: {os.path.abspath(DATA_RAW)}")

Data directory: d:\Documentos\4º UNIVERSIDAD\TFG\Macro-Aviation-Tourism-Modeling\data\raw


### 0.1 File Catalogue

Every raw CSV is registered below, grouped by thematic block. The catalogue serves as the single source of truth for file paths across the project.

In [60]:
# =============================================================
# FILE CATALOGUE — one entry per raw dataset
# =============================================================

FILES = {
    # ---- TOURISM (target + supplementary context) ----
    'tour_occ_nim': os.path.join(DATA_RAW, 'tourism',
        'tour_occ_nights_accommodation_PL_2003-.csv'),
    'tour_cap_nat': os.path.join(DATA_RAW, 'tourism',
        'tour_cap_nat__tourism-infraestructure.csv'),
    'tour_lfsq6r2': os.path.join(DATA_RAW, 'tourism',
        'tour_lfsq6r2__employment-tourism-industries.csv'),

    # ---- ECONOMIC / PRICE ----
    'prc_hicp_midx': os.path.join(DATA_RAW, 'economic',
        'prc_hicp_midx__relevant-countries-2011-.csv'),
    'prc_hicp_aind': os.path.join(DATA_RAW, 'economic',
        'prc_hicp_aind__specific-inflation.csv'),
    'ert_bil_eur_m': os.path.join(DATA_RAW, 'economic',
        'ert_bil_eur_m__exchange-rates-eur-pln-usd-2003-.csv'),
    'namq_10_gdp_real': os.path.join(DATA_RAW, 'economic',
        'namq_10_gdp__real-gdp-CLV-2010.csv'),
    'namq_10_gdp_nom': os.path.join(DATA_RAW, 'economic',
        'namq_10_gdp__market_prices.csv'),
    'prc_ppp_ind': os.path.join(DATA_RAW, 'economic',
        'prc_ppp_ind__price-level-indices.csv'),
    'owid_gdp': os.path.join(DATA_RAW, 'economic',
        'owid-gdp-world-regions-stacked-area.csv'),

    # ---- SENTIMENT / DEMOGRAPHIC ----
    'ei_bsco_m': os.path.join(DATA_RAW, 'demographic',
        'ei_bsco_m__consumer-conf-indicator.csv'),
    'demo_pjan': os.path.join(DATA_RAW, 'demographic',
        'demo_pjan__population-eur-990-25.csv'),
    'earn_nt_net': os.path.join(DATA_RAW, 'demographic',
        'earn_nt_net__annual-net-earnings.csv'),

    # ---- TRANSPORT / AVIATION ----
    'avia_tf_aca': os.path.join(DATA_RAW, 'transport',
        'avia_tf_aca__seats-flights-pass-PL-2010-.csv'),
    'avia_paoc': os.path.join(DATA_RAW, 'transport',
        'avia_paoc__passengers-countries.csv'),
    'avia_tf_apal': os.path.join(DATA_RAW, 'transport',
        'avia_tf_apal_passengers_airports_PL-2003-.csv'),
}

# --- FILE EXISTENCE CHECK ---
print(f"{'Dataset':<20} {'File':<58} {'Status'}")
print('=' * 95)
for key, path in FILES.items():
    fname = os.path.basename(path)
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        status = f'OK ({size_kb:,.0f} KB)'
    else:
        status = 'MISSING'
    print(f'{key:<20} {fname:<58} {status}')

Dataset              File                                                       Status
tour_occ_nim         tour_occ_nights_accommodation_PL_2003-.csv                 OK (1,920 KB)
tour_cap_nat         tour_cap_nat__tourism-infraestructure.csv                  OK (368 KB)
tour_lfsq6r2         tour_lfsq6r2__employment-tourism-industries.csv            OK (1,906 KB)
prc_hicp_midx        prc_hicp_midx__relevant-countries-2011-.csv                OK (465 KB)
prc_hicp_aind        prc_hicp_aind__specific-inflation.csv                      OK (183 KB)
ert_bil_eur_m        ert_bil_eur_m__exchange-rates-eur-pln-usd-2003-.csv        OK (177 KB)
namq_10_gdp_real     namq_10_gdp__real-gdp-CLV-2010.csv                         OK (286 KB)
namq_10_gdp_nom      namq_10_gdp__market_prices.csv                             OK (48 KB)
prc_ppp_ind          prc_ppp_ind__price-level-indices.csv                       OK (141 KB)
owid_gdp             owid-gdp-world-regions-stacked-area.csv                    OK

### 0.2 Inspection Helper Function

The `inspect_dataset()` utility loads a CSV and prints a comprehensive structural summary: dimensions, column types, unique values of categorical filters, temporal coverage, and a basic numeric summary of `OBS_VALUE`. It is called once per dataset in the sections that follow.

In [61]:
def inspect_dataset(path, title):
    """
    Load a CSV and print a comprehensive structural summary.
    
    Parameters
    ----------
    path : str
        Path to the CSV file.
    title : str
        Display title for the output header.
    
    Returns
    -------
    pd.DataFrame or None
        The loaded DataFrame, or None if the file is missing.
    """
    if not os.path.exists(path):
        print(f"\n{'=' * 70}")
        print(f"  {title} — FILE NOT FOUND")
        print(f"  Expected: {path}")
        print(f"{'=' * 70}")
        return None

    df = pd.read_csv(path)

    print(f"\n{'=' * 70}")
    print(f"  {title}")
    print(f"{'=' * 70}")
    print(f"  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")
    print()

    # --- Column overview ---
    print('  COLUMNS:')
    for col in df.columns:
        dtype   = df[col].dtype
        nunique = df[col].nunique()
        nulls   = df[col].isna().sum()
        pct     = nulls / len(df) * 100
        sample  = df[col].dropna().iloc[0] if not df[col].dropna().empty else 'N/A'
        print(f'    {col:<25} {str(dtype):<10} '
              f'{nunique:>6} unique  {nulls:>5} null ({pct:>5.1f}%)  '
              f'sample: {str(sample)[:40]}')
    print()

    # --- Categorical columns: enumerate unique values ---
    skip = {'TIME_PERIOD', 'OBS_FLAG', 'DATAFLOW', 'LAST UPDATE', 'CONF_STATUS'}
    for col in df.columns:
        if df[col].dtype == 'object' and col not in skip:
            n = df[col].nunique()
            vals = sorted(df[col].dropna().unique().astype(str))
            if n <= 30:
                print(f'  {col} ({n} values): {vals}')
            else:
                print(f'  {col} ({n} values): [showing first 15] {vals[:15]}')
    print()

    # --- Temporal coverage ---
    if 'TIME_PERIOD' in df.columns:
        print(f'  TIME_PERIOD: {df["TIME_PERIOD"].min()} -> '
              f'{df["TIME_PERIOD"].max()}  '
              f'({df["TIME_PERIOD"].nunique()} periods)')

    # --- Numeric summary of OBS_VALUE ---
    if 'OBS_VALUE' in df.columns:
        print(f'  OBS_VALUE: min={df["OBS_VALUE"].min()}, '
              f'max={df["OBS_VALUE"].max()}, '
              f'mean={df["OBS_VALUE"].mean():.2f}, '
              f'null={df["OBS_VALUE"].isna().sum()}')
    print()
    return df

### 0.3 Country-Name Mapping

Eurostat datasets use full country names (e.g. `'Germany'`) rather than ISO codes. The mapping below allows us to verify coverage using the standard two-letter codes used throughout the thesis.

In [62]:
# Standard mapping: ISO-2 code -> Eurostat full name
COUNTRY_MAP = {
    'AT': 'Austria',
    'CZ': 'Czechia',
    'DE': 'Germany',
    'EL': 'Greece',
    'ES': 'Spain',
    'FR': 'France',
    'HR': 'Croatia',
    'HU': 'Hungary',
    'IT': 'Italy',
    'NL': 'Netherlands',
    'PL': 'Poland',
    'SE': 'Sweden',
    'UK': 'United Kingdom',
}

# Reverse mapping for quick lookup
NAME_TO_CODE = {v: k for k, v in COUNTRY_MAP.items()}


def check_country_coverage(df, needed_codes, label=''):
    """
    Check which of the needed countries are present in a DataFrame's 'geo' column.
    Handles both ISO codes and full names.
    """
    if df is None or 'geo' not in df.columns:
        print('  Cannot check — DataFrame is None or has no geo column.')
        return
    available = set(df['geo'].unique())
    print(f'Country coverage{" — " + label if label else ""}:')
    for code in needed_codes:
        name = COUNTRY_MAP.get(code, code)
        # Try both the code and the full name
        found_as = None
        if code in available:
            found_as = code
        elif name in available:
            found_as = name
        if found_as:
            subset = df[df['geo'] == found_as]
            tmin = subset['TIME_PERIOD'].min() if 'TIME_PERIOD' in df.columns else '?'
            tmax = subset['TIME_PERIOD'].max() if 'TIME_PERIOD' in df.columns else '?'
            print(f'  {code} ({name}): FOUND — {tmin} -> {tmax} ({len(subset)} obs)')
        else:
            print(f'  {code} ({name}): MISSING')
    print()

---

## 1. Tourism Block

This block contains the **dependent variable** (overnight stays) as well as two supplementary contextual datasets (accommodation capacity and tourism employment) that are used for descriptive purposes only.

### 1.1 Overnight Stays — `tour_occ_nim` (Dependent Variable)

This is the **target variable** of the thesis: monthly overnight stays by foreign tourists in tourist accommodation establishments in Poland, sourced from Eurostat table `tour_occ_nim`.

**Critical clarification on `c_resid`:**  
The column `c_resid` (country of residence) in this dataset distinguishes between **domestic** and **foreign** guests at the aggregate level. It does **not** provide a breakdown by specific origin country. The available values are:

- `'Domestic country'` — guests residing in Poland
- `'Foreign country'` — guests residing outside Poland (aggregate of all foreign origins)
- `'Total'` — domestic + foreign combined

This means we model **total foreign overnight stays in Poland** as a single aggregate series, rather than separate series per origin country. The origin-country dimension enters the model through the **exogenous variables** (GDP, HICP, consumer confidence, exchange rates, air connectivity) measured at the origin-country level.

The `avia_paoc` dataset (Section 4.2) provides country-level air passenger flows and can be used for descriptive validation of the relative importance of each origin market.

In [63]:
df_nights = inspect_dataset(FILES['tour_occ_nim'], 'OVERNIGHT STAYS — tour_occ_nim')


  OVERNIGHT STAYS — tour_occ_nim
  Shape: 9,601 rows × 11 columns
  Memory: 5.7 MB

  COLUMNS:
    DATAFLOW                  object          1 unique      0 null (  0.0%)  sample: ESTAT:TOUR_OCC_NIM(1.0)
    LAST UPDATE               object          1 unique      0 null (  0.0%)  sample: 02/02/26 23:00:00
    freq                      object          1 unique      0 null (  0.0%)  sample: Monthly
    c_resid                   object          3 unique      0 null (  0.0%)  sample: Domestic country
    unit                      object          4 unique      0 null (  0.0%)  sample: Number
    nace_r2                   object          5 unique      0 null (  0.0%)  sample: Hotels and similar accommodation
    geo                       object          2 unique      0 null (  0.0%)  sample: European Union - 28 countries (2013-2020
    TIME_PERIOD               object        275 unique      0 null (  0.0%)  sample: 2014-01
    OBS_VALUE                 float64      7484 unique      0 null (

In [64]:
# --- DETAILED VALUE COUNTS FOR KEY FILTER COLUMNS ---
if df_nights is not None:
    print('=' * 55)
    print('DETAILED VALUE COUNTS — FILTER COLUMNS')
    print('=' * 55)
    for col in ['unit', 'c_resid', 'nace_r2', 'geo']:
        if col in df_nights.columns:
            print(f'\n--- {col} ---')
            for val, cnt in df_nights[col].value_counts().items():
                print(f'  {val:<60} {cnt:>6} rows')

DETAILED VALUE COUNTS — FILTER COLUMNS

--- unit ---
  Number                                                         4269 rows
  Percentage change compared to same period in previous year     4123 rows
  Percentage change compared to same month in 2019               1065 rows
  Percentage change compared to same period two years ago         144 rows

--- c_resid ---
  Total                                                          3405 rows
  Foreign country                                                3403 rows
  Domestic country                                               2793 rows

--- nace_r2 ---
  Hotels and similar accommodation                               2127 rows
  Hotels; holiday and other short-stay accommodation; camping grounds, recreational vehicle parks and trailer parks   2127 rows
  Holiday and other short-stay accommodation; camping grounds, recreational vehicle parks and trailer parks   2127 rows
  Holiday and other short-stay accommodation                     

In [65]:
# --- VERIFY STRUCTURE OF THE TARGET SERIES ---
# For modelling we need: geo='Poland', c_resid='Foreign country',
# unit='Number', nace_r2 = total accommodation (all types).

if df_nights is not None:
    # Identify the "total accommodation" nace_r2 label
    nace_vals = df_nights['nace_r2'].unique()
    total_accom = [v for v in nace_vals if 'Hotels; holiday' in str(v)]
    print(f'Total accommodation label: {total_accom}')
    print()

    # Filter to the target series
    mask = (
        (df_nights['geo'] == 'Poland') &
        (df_nights['c_resid'] == 'Foreign country') &
        (df_nights['unit'] == 'Number')
    )
    if total_accom:
        mask = mask & (df_nights['nace_r2'] == total_accom[0])

    target = df_nights.loc[mask].sort_values('TIME_PERIOD')
    print(f'Target series (foreign overnight stays in Poland, all accommodation, absolute numbers):')
    print(f'  Rows: {len(target):,}')
    print(f'  Period: {target["TIME_PERIOD"].min()} -> {target["TIME_PERIOD"].max()}')
    print(f'  OBS_VALUE range: {target["OBS_VALUE"].min():,.0f} -> {target["OBS_VALUE"].max():,.0f}')
    print(f'  Missing values: {target["OBS_VALUE"].isna().sum()}')

Total accommodation label: ['Hotels; holiday and other short-stay accommodation; camping grounds, recreational vehicle parks and trailer parks']

Target series (foreign overnight stays in Poland, all accommodation, absolute numbers):
  Rows: 275
  Period: 2003-01 -> 2025-11
  OBS_VALUE range: 83,695 -> 2,453,961
  Missing values: 0


### 1.2 Tourism Infrastructure — `tour_cap_nat` (Supplementary)

In [66]:
df_tour_cap = inspect_dataset(FILES['tour_cap_nat'], 'TOURISM INFRASTRUCTURE — tour_cap_nat')


  TOURISM INFRASTRUCTURE — tour_cap_nat
  Shape: 1,885 rows × 11 columns
  Memory: 1.0 MB

  COLUMNS:
    DATAFLOW                  object          1 unique      0 null (  0.0%)  sample: ESTAT:TOUR_CAP_NAT(1.0)
    LAST UPDATE               object          1 unique      0 null (  0.0%)  sample: 19/02/26 23:00:00
    freq                      object          1 unique      0 null (  0.0%)  sample: Annual
    accomunit                 object          3 unique      0 null (  0.0%)  sample: Bedplaces
    unit                      object          2 unique      0 null (  0.0%)  sample: Number
    nace_r2                   object          5 unique      0 null (  0.0%)  sample: Hotels and similar accommodation
    geo                       object          3 unique      0 null (  0.0%)  sample: Euro area (EA11-1999, EA12-2001, EA13-20
    TIME_PERIOD               int64          35 unique      0 null (  0.0%)  sample: 2001
    OBS_VALUE                 float64      1549 unique      0 null (  0.

> **Usage note:** This annual dataset provides accommodation capacity (establishments, bedrooms, bedplaces) for Poland and selected reference countries. It is used for **descriptive context only** (Chapter 4) and is not included as a model predictor.

### 1.3 Tourism Employment — `tour_lfsq6r2` (Supplementary)

In [67]:
df_tour_emp = inspect_dataset(FILES['tour_lfsq6r2'], 'TOURISM EMPLOYMENT — tour_lfsq6r2')


  TOURISM EMPLOYMENT — tour_lfsq6r2
  Shape: 12,401 rows × 13 columns
  Memory: 8.1 MB

  COLUMNS:
    DATAFLOW                  object          1 unique      0 null (  0.0%)  sample: ESTAT:TOUR_LFSQ6R2(1.0)
    LAST UPDATE               object          1 unique      0 null (  0.0%)  sample: 07/01/26 23:00:00
    freq                      object          1 unique      0 null (  0.0%)  sample: Quarterly
    worktime                  object          3 unique      0 null (  0.0%)  sample: Full-time
    wstatus                   object          2 unique      0 null (  0.0%)  sample: Employed persons
    sex                       object          3 unique      0 null (  0.0%)  sample: Females
    nace_r2                   object          5 unique      0 null (  0.0%)  sample: Air transport
    unit                      object          1 unique      0 null (  0.0%)  sample: Thousand persons
    geo                       object          2 unique      0 null (  0.0%)  sample: Spain
    TIME_PE

> **Usage note:** Quarterly employment data in tourism-related **NACE**<sup>[9](#fn9)</sup> sectors. Used for **descriptive context only** — not suitable as a model predictor due to quarterly frequency, limited country scope, and high proportion of missing values (12.4% in `OBS_VALUE`).

---

## 2. Economic / Price Block

This block contains the variables needed to construct the **relative tourism price** (Song et al., 2010) and the **income proxy** (real GDP) for each origin market.

### 2.1 HICP<sup>[2](#fn2)</sup> Monthly Index — `prc_hicp_midx`

**Role in modelling:** Primary CPI variable for the relative tourism price formula:  
$P_{it} = \frac{\text{CPI}_{PL}}{\text{CPI}_i} \times \frac{\text{EX}_i}{\text{EX}_{PL}}$

We need the **All-items HICP** (index, 2015 = 100) at monthly frequency for Poland and all origin/substitute countries.

In [68]:
df_hicp_m = inspect_dataset(FILES['prc_hicp_midx'], 'HICP MONTHLY — prc_hicp_midx')


  HICP MONTHLY — prc_hicp_midx
  Shape: 4,067 rows × 10 columns
  Memory: 1.9 MB

  COLUMNS:
    DATAFLOW                  object          1 unique      0 null (  0.0%)  sample: ESTAT:PRC_HICP_MIDX(1.0)
    LAST UPDATE               object          1 unique      0 null (  0.0%)  sample: 06/02/26 23:00:00
    freq                      object          1 unique      0 null (  0.0%)  sample: Monthly
    unit                      object          1 unique      0 null (  0.0%)  sample: Index, 2015=100
    coicop                    object          1 unique      0 null (  0.0%)  sample: All-items HICP
    geo                       object         15 unique      0 null (  0.0%)  sample: Austria
    TIME_PERIOD               object        276 unique      0 null (  0.0%)  sample: 2003-01
    OBS_VALUE                 float64      2342 unique      0 null (  0.0%)  sample: 78.71
    OBS_FLAG                  object          1 unique   3803 null ( 93.5%)  sample: d
    CONF_STATUS               float

In [69]:
# --- VERIFY COUNTRY × COICOP COVERAGE ---
# All values should be 'All-items HICP' (single coicop in this extract).
# We check that every required country is present.

needed_hicp = ['AT', 'CZ', 'DE', 'EL', 'ES', 'FR', 'HR', 'HU', 'IT', 'NL', 'PL', 'SE', 'UK']
check_country_coverage(df_hicp_m, needed_hicp, label='HICP monthly')

Country coverage — HICP monthly:
  AT (Austria): FOUND — 2003-01 -> 2025-12 (276 obs)
  CZ (Czechia): FOUND — 2003-01 -> 2025-12 (276 obs)
  DE (Germany): FOUND — 2003-01 -> 2025-12 (276 obs)
  EL (Greece): FOUND — 2003-01 -> 2025-12 (276 obs)
  ES (Spain): FOUND — 2003-01 -> 2025-12 (276 obs)
  FR (France): FOUND — 2003-01 -> 2025-12 (276 obs)
  HR (Croatia): FOUND — 2003-01 -> 2025-12 (276 obs)
  HU (Hungary): FOUND — 2003-01 -> 2025-12 (276 obs)
  IT (Italy): FOUND — 2003-01 -> 2025-12 (276 obs)
  NL (Netherlands): FOUND — 2003-01 -> 2025-12 (276 obs)
  PL (Poland): FOUND — 2003-01 -> 2025-12 (276 obs)
  SE (Sweden): FOUND — 2003-01 -> 2025-12 (276 obs)
  UK (United Kingdom): FOUND — 2003-01 -> 2020-11 (215 obs)



> **Finding:** All 13 required countries are present. The UK series is shorter (ends earlier due to post-Brexit reporting changes), which will need to be handled in Notebook 04 (Feature Engineering). The `coicop`<sup>[7](#fn7)</sup> column is pre-filtered to `'All-items HICP'` and the `unit` is `'Index, 2015=100'` — no further filtering is needed.

### 2.2 HICP Annual Index — `prc_hicp_aind` (Descriptive Only)

**Role:** Sector-specific inflation indices (Restaurants & Hotels, Accommodation) for the descriptive chapter. **Not used in models** — the monthly index (`prc_hicp_midx`) is the modelling variable.

In [70]:
df_hicp_a = inspect_dataset(FILES['prc_hicp_aind'], 'HICP ANNUAL — prc_hicp_aind')


  HICP ANNUAL — prc_hicp_aind
  Shape: 1,476 rows × 10 columns
  Memory: 0.7 MB

  COLUMNS:
    DATAFLOW                  object          1 unique      0 null (  0.0%)  sample: ESTAT:PRC_HICP_AIND(1.0)
    LAST UPDATE               object          1 unique      0 null (  0.0%)  sample: 06/02/26 23:00:00
    freq                      object          1 unique      0 null (  0.0%)  sample: Annual
    unit                      object          1 unique      0 null (  0.0%)  sample: Annual average index
    coicop                    object          5 unique      0 null (  0.0%)  sample: All-items HICP
    geo                       object         15 unique      0 null (  0.0%)  sample: Austria
    TIME_PERIOD               int64          23 unique      0 null (  0.0%)  sample: 2003
    OBS_VALUE                 float64      1160 unique      1 null (  0.1%)  sample: 79.05
    OBS_FLAG                  object          3 unique   1415 null ( 95.9%)  sample: d
    CONF_STATUS               objec

### 2.3 Exchange Rates — `ert_bil_eur_m`

**Role in modelling:** Monthly bilateral exchange rates (national currency per EUR) for non-Eurozone countries. Required for the relative tourism price formula.

**Required currencies:**  
- PLN (Polish zloty) — destination  
- GBP (Pound sterling), SEK (Swedish krona), CZK (Czech koruna), HUF (Hungarian forint) — non-Eurozone origins  
- USD (US dollar) — reference  

For Eurozone-based origins (DE, FR, NL, IT, ES, AT, EL, HR), the exchange rate ratio in the price formula cancels to 1.

In [71]:
df_exr = inspect_dataset(FILES['ert_bil_eur_m'], 'EXCHANGE RATES — ert_bil_eur_m')


  EXCHANGE RATES — ert_bil_eur_m
  Shape: 1,668 rows × 10 columns
  Memory: 0.7 MB

  COLUMNS:
    DATAFLOW                  object          1 unique      0 null (  0.0%)  sample: ESTAT:ERT_BIL_EUR_M(1.0)
    LAST UPDATE               object          1 unique      0 null (  0.0%)  sample: 03/03/26 11:00:00
    freq                      object          1 unique      0 null (  0.0%)  sample: Monthly
    statinfo                  object          1 unique      0 null (  0.0%)  sample: Average
    unit                      object          1 unique      0 null (  0.0%)  sample: National currency
    currency                  object          6 unique      0 null (  0.0%)  sample: Czech koruna
    TIME_PERIOD               object        278 unique      0 null (  0.0%)  sample: 2003-01
    OBS_VALUE                 float64      1641 unique      0 null (  0.0%)  sample: 31.489
    OBS_FLAG                  float64         0 unique   1668 null (100.0%)  sample: N/A
    CONF_STATUS               

In [72]:
# --- VERIFY REQUIRED CURRENCIES ---
if df_exr is not None and 'currency' in df_exr.columns:
    needed_curr = {
        'PLN': 'Polish zloty',
        'GBP': 'Pound sterling',
        'SEK': 'Swedish krona',
        'CZK': 'Czech koruna',
        'HUF': 'Hungarian forint',
        'USD': 'US dollar',
    }
    available_curr = set(df_exr['currency'].unique())
    print('Currency verification:')
    for code, name in needed_curr.items():
        # Match by full name (Eurostat uses full names, not ISO codes)
        match = name in available_curr or code in available_curr
        label = name if name in available_curr else code
        if match:
            subset = df_exr[df_exr['currency'] == label]
            print(f'  {code} ({name}): FOUND — '
                  f'{subset["TIME_PERIOD"].min()} -> {subset["TIME_PERIOD"].max()}')
        else:
            print(f'  {code} ({name}): NOT FOUND')

Currency verification:
  PLN (Polish zloty): FOUND — 2003-01 -> 2026-02
  GBP (Pound sterling): FOUND — 2003-01 -> 2026-02
  SEK (Swedish krona): FOUND — 2003-01 -> 2026-02
  CZK (Czech koruna): FOUND — 2003-01 -> 2026-02
  HUF (Hungarian forint): FOUND — 2003-01 -> 2026-02
  USD (US dollar): FOUND — 2003-01 -> 2026-02


> **Finding:** All six required currencies are present with full temporal coverage from 2003 onwards. The `currency` column uses full names (e.g. `'Polish zloty'`); downstream notebooks will map these to ISO codes for consistency.

### 2.4 Real GDP — `namq_10_gdp` (Chain-Linked Volumes<sup>[3](#fn3)</sup>, 2010)

**Role in modelling:** Income proxy for origin-country tourists. Quarterly series (seasonally and calendar adjusted) that will be **interpolated to monthly** frequency via **cubic spline**<sup>[1](#fn1)</sup> in Notebook 04.

We need GDP data for all origin markets plus Poland and the substitute destinations.

In [73]:
df_gdp = inspect_dataset(FILES['namq_10_gdp_real'], 'REAL GDP — namq_10_gdp (CLV 2010)')


  REAL GDP — namq_10_gdp (CLV 2010)
  Shape: 1,459 rows × 11 columns
  Memory: 0.9 MB

  COLUMNS:
    DATAFLOW                  object          1 unique      0 null (  0.0%)  sample: ESTAT:NAMQ_10_GDP(1.0)
    LAST UPDATE               object          1 unique      0 null (  0.0%)  sample: 13/03/26 23:00:00
    freq                      object          1 unique      0 null (  0.0%)  sample: Quarterly
    unit                      object          1 unique      0 null (  0.0%)  sample: Chain linked volumes (2010), million eur
    s_adj                     object          1 unique      0 null (  0.0%)  sample: Seasonally and calendar adjusted data
    na_item                   object          1 unique      0 null (  0.0%)  sample: Gross domestic product at market prices
    geo                       object         12 unique      0 null (  0.0%)  sample: Austria
    TIME_PERIOD               object        124 unique      0 null (  0.0%)  sample: 1995-Q1
    OBS_VALUE                 float

In [74]:
needed_gdp = ['AT', 'CZ', 'DE', 'ES', 'FR', 'HR', 'HU', 'IT', 'NL', 'PL', 'SE', 'UK']
check_country_coverage(df_gdp, needed_gdp, label='Real GDP')

Country coverage — Real GDP:
  AT (Austria): FOUND — 1995-Q1 -> 2025-Q4 (124 obs)
  CZ (Czechia): FOUND — 1995-Q1 -> 2025-Q4 (124 obs)
  DE (Germany): FOUND — 1995-Q1 -> 2025-Q4 (124 obs)
  ES (Spain): FOUND — 1995-Q1 -> 2025-Q4 (124 obs)
  FR (France): FOUND — 1995-Q1 -> 2025-Q4 (124 obs)
  HR (Croatia): FOUND — 1995-Q1 -> 2025-Q4 (124 obs)
  HU (Hungary): FOUND — 1995-Q1 -> 2025-Q4 (124 obs)
  IT (Italy): FOUND — 1996-Q1 -> 2025-Q4 (120 obs)
  NL (Netherlands): FOUND — 1996-Q1 -> 2025-Q4 (120 obs)
  PL (Poland): FOUND — 1995-Q1 -> 2025-Q4 (124 obs)
  SE (Sweden): FOUND — 1995-Q1 -> 2025-Q4 (124 obs)
  UK (United Kingdom): FOUND — 1995-Q1 -> 2020-Q3 (103 obs)



> **Finding:** All 12 countries are present. The UK series ends at **2020-Q3** due to post-Brexit Eurostat reporting changes — this gap will require supplementation from an alternative source (e.g. IMF WEO or OECD) in Notebook 04. Italy and the Netherlands start from 1996-Q1 rather than 1995-Q1, but this does not affect our study period (2011–2025).

### 2.5 Nominal GDP — `namq_10_gdp` (Current Prices, Poland Only)

In [75]:
df_gdp_nom = inspect_dataset(FILES['namq_10_gdp_nom'], 'NOMINAL GDP — namq_10_gdp (market prices)')


  NOMINAL GDP — namq_10_gdp (market prices)
  Shape: 246 rows × 11 columns
  Memory: 0.1 MB

  COLUMNS:
    DATAFLOW                  object          1 unique      0 null (  0.0%)  sample: ESTAT:NAMQ_10_GDP(1.0)
    LAST UPDATE               object          1 unique      0 null (  0.0%)  sample: 20/02/26 23:00:00
    freq                      object          1 unique      0 null (  0.0%)  sample: Quarterly
    unit                      object          2 unique      0 null (  0.0%)  sample: Current prices, million euro
    s_adj                     object          1 unique      0 null (  0.0%)  sample: Seasonally and calendar adjusted data
    na_item                   object          1 unique      0 null (  0.0%)  sample: Gross domestic product at market prices
    geo                       object          1 unique      0 null (  0.0%)  sample: Poland
    TIME_PERIOD               object        123 unique      0 null (  0.0%)  sample: 1995-Q1
    OBS_VALUE                 float64     

> **Usage note:** This extract contains Poland's nominal GDP only (current prices, million EUR and million PLN). Used for **descriptive context** and potential normalisation — not a direct model predictor.

### 2.6 Purchasing Power Parities<sup>[10](#fn10)</sup> — `prc_ppp_ind`

In [76]:
df_ppp = inspect_dataset(FILES['prc_ppp_ind'], 'PPP INDICES — prc_ppp_ind')


  PPP INDICES — prc_ppp_ind
  Shape: 1,056 rows × 10 columns
  Memory: 0.4 MB

  COLUMNS:
    DATAFLOW                  object          1 unique      0 null (  0.0%)  sample: ESTAT:PRC_PPP_IND(1.0)
    LAST UPDATE               object          1 unique      0 null (  0.0%)  sample: 10/07/25 11:00:00
    freq                      object          1 unique      0 null (  0.0%)  sample: Annual
    na_item                   object          1 unique      0 null (  0.0%)  sample: Price level indices (EU27_2020=100)
    ppp_cat                   object          1 unique      0 null (  0.0%)  sample: Actual individual consumption
    geo                       object         36 unique      0 null (  0.0%)  sample: Albania
    TIME_PERIOD               int64          30 unique      0 null (  0.0%)  sample: 1997
    OBS_VALUE                 float64       710 unique      0 null (  0.0%)  sample: 27.7
    OBS_FLAG                  float64         0 unique   1056 null (100.0%)  sample: N/A
    CONF

> **Usage note:** Annual price-level indices (EU27 2020 = 100) for actual individual consumption. Provides a complementary perspective on price competitiveness alongside the HICP-based relative price variable. Used for **descriptive analysis** in Chapter 4.

### 2.7 OWID GDP — `owid-gdp-world-regions-stacked-area` (Supplementary)

In [77]:
df_owid = inspect_dataset(FILES['owid_gdp'], 'OWID GDP — owid-gdp-world-regions-stacked-area')


  OWID GDP — owid-gdp-world-regions-stacked-area
  Shape: 16,143 rows × 5 columns
  Memory: 2.4 MB

  COLUMNS:
    Entity                    object        178 unique      0 null (  0.0%)  sample: Afghanistan
    Code                      object        169 unique    234 null (  1.4%)  sample: AFG
    Year                      int64         208 unique      0 null (  0.0%)  sample: 1950
    Gross domestic product (GDP) int64       16138 unique      0 null (  0.0%)  sample: 9421400000
    900795-annotations        object          1 unique  16122 null ( 99.9%)  sample: United States, Canada, Australia and New

  Entity (178 values): [showing first 15] ['Afghanistan', 'Albania', 'Algeria', 'Angola', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Benin']
  Code (169 values): [showing first 15] ['AFG', 'AGO', 'ALB', 'ARE', 'ARG', 'ARM', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR']
  900795-annotations

> **Usage note:** This Our World in Data extract provides long-run GDP context at the world/regional level. Used for **descriptive background** only — not a model input.

---

## 3. Sentiment & Demographic Block

Variables capturing consumer sentiment and socio-economic context in origin markets.

### 3.1 Consumer Confidence Indicator — `ei_bsco_m`

**Role in modelling:** Monthly sentiment variable capturing consumers' willingness to spend on discretionary items such as tourism. Seasonally adjusted balance indicator.

In [78]:
df_cci = inspect_dataset(FILES['ei_bsco_m'], 'CONSUMER CONFIDENCE — ei_bsco_m')


  CONSUMER CONFIDENCE — ei_bsco_m
  Shape: 6,411 rows × 11 columns
  Memory: 3.4 MB

  COLUMNS:
    DATAFLOW                  object          1 unique      0 null (  0.0%)  sample: ESTAT:EI_BSCO_M(1.0)
    LAST UPDATE               object          1 unique      0 null (  0.0%)  sample: 26/02/26 11:00:00
    freq                      object          1 unique      0 null (  0.0%)  sample: Monthly
    indic                     object          1 unique      0 null (  0.0%)  sample: Consumer confidence indicator
    s_adj                     object          1 unique      0 null (  0.0%)  sample: Seasonally adjusted data, not calendar a
    unit                      object          1 unique      0 null (  0.0%)  sample: Balance
    geo                       object         26 unique      0 null (  0.0%)  sample: Austria
    TIME_PERIOD               object        254 unique      0 null (  0.0%)  sample: 2005-01
    OBS_VALUE                 float64       679 unique      0 null (  0.0%)  samp

In [79]:
needed_cci = ['AT', 'CZ', 'DE', 'ES', 'FR', 'IT', 'NL', 'SE', 'PL', 'HU', 'UK']
check_country_coverage(df_cci, needed_cci, label='Consumer confidence')

Country coverage — Consumer confidence:
  AT (Austria): FOUND — 2005-01 -> 2026-02 (254 obs)
  CZ (Czechia): FOUND — 2005-01 -> 2026-02 (254 obs)
  DE (Germany): FOUND — 2005-01 -> 2026-02 (254 obs)
  ES (Spain): FOUND — 2005-01 -> 2026-02 (254 obs)
  FR (France): FOUND — 2005-01 -> 2026-02 (254 obs)
  IT (Italy): FOUND — 2005-01 -> 2026-02 (253 obs)
  NL (Netherlands): FOUND — 2005-01 -> 2026-02 (254 obs)
  SE (Sweden): FOUND — 2005-01 -> 2026-02 (254 obs)
  PL (Poland): FOUND — 2005-01 -> 2026-02 (254 obs)
  HU (Hungary): FOUND — 2005-01 -> 2026-02 (254 obs)
  UK (United Kingdom): MISSING



> **Finding:** The UK is **missing** from this Eurostat dataset (the UK stopped contributing to EU business surveys after Brexit). An alternative source (e.g. GfK UK Consumer Confidence) would be needed if UK-specific sentiment was required. All other origin markets are present with monthly data from 2005-01 onwards.

### 3.2 Population — `demo_pjan`

**Role:** Annual total population on 1 January. Used as a **contextual/normalisation** variable (e.g. overnight stays per capita) rather than a direct model predictor.

In [80]:
df_pop = inspect_dataset(FILES['demo_pjan'], 'POPULATION — demo_pjan')


  POPULATION — demo_pjan
  Shape: 1,629 rows × 11 columns
  Memory: 0.7 MB

  COLUMNS:
    DATAFLOW                  object          1 unique      0 null (  0.0%)  sample: ESTAT:DEMO_PJAN(1.0)
    LAST UPDATE               object          1 unique      0 null (  0.0%)  sample: 13/02/26 11:00:00
    freq                      object          1 unique      0 null (  0.0%)  sample: Annual
    unit                      object          1 unique      0 null (  0.0%)  sample: Number
    age                       object          1 unique      0 null (  0.0%)  sample: Total
    sex                       object          1 unique      0 null (  0.0%)  sample: Total
    geo                       object         52 unique      0 null (  0.0%)  sample: Andorra
    TIME_PERIOD               int64          36 unique      0 null (  0.0%)  sample: 1990
    OBS_VALUE                 int64        1593 unique      0 null (  0.0%)  sample: 50887
    OBS_FLAG                  object          6 unique   1552 n

### 3.3 Net Earnings — `earn_nt_net`

**Role:** Annual net earnings for a single earner at 100% of average wage. Provides a complementary income measure alongside GDP. **Descriptive use only** due to annual frequency.

In [81]:
df_earn = inspect_dataset(FILES['earn_nt_net'], 'NET EARNINGS — earn_nt_net')


  NET EARNINGS — earn_nt_net
  Shape: 3,330 rows × 11 columns
  Memory: 1.7 MB

  COLUMNS:
    DATAFLOW                  object          1 unique      0 null (  0.0%)  sample: ESTAT:EARN_NT_NET(1.0)
    LAST UPDATE               object          1 unique      0 null (  0.0%)  sample: 09/02/26 23:00:00
    freq                      object          1 unique      0 null (  0.0%)  sample: Annual
    currency                  object          2 unique      0 null (  0.0%)  sample: Euro
    estruct                   object          1 unique      0 null (  0.0%)  sample: Net earning
    ecase                     object          2 unique      0 null (  0.0%)  sample: Two-earner couple without children, both
    geo                       object         39 unique      0 null (  0.0%)  sample: Austria
    TIME_PERIOD               int64          25 unique      0 null (  0.0%)  sample: 2000
    OBS_VALUE                 float64      2456 unique      0 null (  0.0%)  sample: 41051.73
    OBS_FLAG   

---

## 4. Transport / Aviation Block

Air connectivity variables. **Endogeneity<sup>[5](#fn5)</sup> caveat (warning):** airlines set capacity partly based on demand expectations. These variables are used either with a 6–12 month lag (capacity is predetermined via **IATA slot allocation**<sup>[4](#fn4)</sup>), as a structural/descriptive variable, or through the genuinely exogenous dimension of **LCC**<sup>[6](#fn6)</sup> entry (Wizz Air, Ryanair). See `docs/PROJECT_STRUCTURE.md` for the full discussion.

### 4.1 Air Capacity — `avia_tf_aca`

**Role in modelling:** Monthly airline seats available (`'Passengers seats available'`) at Polish airports. Primary measure of air connectivity / accessibility.

This dataset also contains flights and passengers on board, but the **seats available** measure is the most relevant for the supply-side connectivity argument.

In [82]:
df_avia = inspect_dataset(FILES['avia_tf_aca'], 'AVIATION CAPACITY — avia_tf_aca')


  AVIATION CAPACITY — avia_tf_aca
  Shape: 6,574 rows × 12 columns
  Memory: 3.7 MB

  COLUMNS:
    DATAFLOW                  object          1 unique      0 null (  0.0%)  sample: ESTAT:AVIA_TF_ACA(1.0)
    LAST UPDATE               object          1 unique      0 null (  0.0%)  sample: 25/02/26 23:00:00
    freq                      object          1 unique      0 null (  0.0%)  sample: Monthly
    unit                      object          3 unique      0 null (  0.0%)  sample: Flight
    tra_meas                  object          3 unique      0 null (  0.0%)  sample: Commercial passenger air flights
    tra_cov                   object          1 unique      0 null (  0.0%)  sample: Total transport
    aircraft                  object          1 unique      0 null (  0.0%)  sample: Total
    rep_airp                  object         13 unique      0 null (  0.0%)  sample: BYDGOSZCZ/SZWEDEROWO airport
    TIME_PERIOD               object        190 unique      0 null (  0.0%)  sample

In [83]:
# --- DETAIL: Transport measures and airports ---
if df_avia is not None:
    for col in ['tra_meas', 'rep_airp']:
        if col in df_avia.columns:
            print(f'--- {col} ({df_avia[col].nunique()} values) ---')
            for val, cnt in df_avia[col].value_counts().items():
                print(f'  {val:<45} {cnt:>6} rows')
            print()

--- tra_meas (3 values) ---
  Passengers on board                             2193 rows
  Passengers seats available                      2193 rows
  Commercial passenger air flights                2188 rows

--- rep_airp (13 values) ---
  GDANSK IM LECHA WALESY airport                   570 rows
  KRAKOW/BALICE airport                            570 rows
  KATOWICE/PYRZOWICE airport                       570 rows
  WARSZAWA/CHOPINA airport                         570 rows
  WROCLAW/STRACHOWICE airport                      570 rows
  POZNAN/LAWICA airport                            568 rows
  LODZ/LUBLINEK airport                            534 rows
  RZESZOW/JASIONKA airport                         534 rows
  SZCZECIN/GOLENIOW airport                        534 rows
  BYDGOSZCZ/SZWEDEROWO airport                     531 rows
  WARSZAWA/MODLIN airport                          425 rows
  LUBLIN airport                                   424 rows
  OLSZTYN-MAZURY airport                  

> **Finding:** Three transport measures are available: seats available, passengers on board, and commercial flights — all at the individual airport level (13 Polish airports). For modelling, we will aggregate seats across all airports to obtain a national-level monthly series.

### 4.2 Air Passengers by Country Pair — `avia_paoc`

**Role:** Descriptive and validation only — **not used as a predictor** (near-tautological with tourism demand).

This dataset shows monthly passenger flows between Poland and specific partner countries, allowing us to validate the relative importance of each origin market.

In [84]:
df_pax = inspect_dataset(FILES['avia_paoc'], 'AVIATION PASSENGERS — avia_paoc')


  AVIATION PASSENGERS — avia_paoc
  Shape: 3,408 rows × 12 columns
  Memory: 1.9 MB

  COLUMNS:
    DATAFLOW                  object          1 unique      0 null (  0.0%)  sample: ESTAT:AVIA_PAOC(1.0)
    LAST UPDATE               object          1 unique      0 null (  0.0%)  sample: 25/02/26 23:00:00
    freq                      object          1 unique      0 null (  0.0%)  sample: Monthly
    unit                      object          1 unique      0 null (  0.0%)  sample: Passenger
    tra_meas                  object          1 unique      0 null (  0.0%)  sample: Passengers on board
    tra_cov                   object          1 unique      0 null (  0.0%)  sample: Total transport
    schedule                  object          1 unique      0 null (  0.0%)  sample: Total
    geo                       object         13 unique      0 null (  0.0%)  sample: Austria
    TIME_PERIOD               object        276 unique      0 null (  0.0%)  sample: 2003-01
    OBS_VALUE          

In [85]:
needed_pax = ['AT', 'CZ', 'DE', 'ES', 'FR', 'EL', 'HR', 'HU', 'IT', 'NL', 'SE', 'UK']
check_country_coverage(df_pax, needed_pax, label='Air passengers by country')

Country coverage — Air passengers by country:
  AT (Austria): FOUND — 2003-01 -> 2025-11 (275 obs)
  CZ (Czechia): FOUND — 2004-04 -> 2025-10 (259 obs)
  DE (Germany): FOUND — 2003-01 -> 2025-12 (276 obs)
  ES (Spain): FOUND — 2003-01 -> 2025-12 (276 obs)
  FR (France): FOUND — 2003-01 -> 2025-07 (271 obs)
  EL (Greece): FOUND — 2003-01 -> 2025-08 (272 obs)
  HR (Croatia): FOUND — 2008-01 -> 2025-11 (215 obs)
  HU (Hungary): FOUND — 2003-01 -> 2025-12 (276 obs)
  IT (Italy): FOUND — 2003-01 -> 2025-09 (273 obs)
  NL (Netherlands): FOUND — 2003-01 -> 2025-12 (276 obs)
  SE (Sweden): FOUND — 2003-01 -> 2025-08 (272 obs)
  UK (United Kingdom): FOUND — 2003-01 -> 2020-01 (205 obs)



### 4.3 Airport-Level Passengers — `avia_tf_apal`

In [ ]:
df_apal = inspect_dataset(FILES['avia_tf_apal'], 'AIRPORT PASSENGERS — avia_tf_apal')


  AIRPORT PASSENGERS — avia_tf_apal
  Shape: 1,200 rows × 11 columns
  Memory: 0.6 MB

  COLUMNS:
    DATAFLOW                  object          1 unique      0 null (  0.0%)  sample: ESTAT:AVIA_TF_APAL(1.0)
    LAST UPDATE               object          1 unique      0 null (  0.0%)  sample: 16/12/25 23:00:00
    freq                      object          1 unique      0 null (  0.0%)  sample: Monthly
    unit                      object          1 unique      0 null (  0.0%)  sample: Passenger
    tra_meas                  object          1 unique      0 null (  0.0%)  sample: Passengers carried
    airline                   object          1 unique      0 null (  0.0%)  sample: All airlines
    rep_airp                  object         15 unique      0 null (  0.0%)  sample: BYDGOSZCZ/SZWEDEROWO airport
    TIME_PERIOD               object        260 unique      0 null (  0.0%)  sample: 2008-01
    OBS_VALUE                 int64        1192 unique      0 null (  0.0%)  sample: 17441
 

> **Usage note:** Monthly total passengers carried per Polish airport. Useful for descriptive analysis of airport-level trends (e.g. growth of regional airports, Modlin as a LCC hub). Not used directly in modelling.

---

## 5. Data Inventory Summary

### 5.1 Consolidated Inventory Table

The table below summarises every dataset inspected in this notebook: its dimensions, temporal coverage, number of distinct countries (where applicable), and missing-value status.

In [87]:
# =============================================================
# CONSOLIDATED INVENTORY TABLE
# =============================================================

all_datasets = {
    'tour_occ_nim':     df_nights,
    'tour_cap_nat':     df_tour_cap,
    'tour_lfsq6r2':     df_tour_emp,
    'prc_hicp_midx':    df_hicp_m,
    'prc_hicp_aind':    df_hicp_a,
    'ert_bil_eur_m':    df_exr,
    'namq_10_gdp_real': df_gdp,
    'namq_10_gdp_nom':  df_gdp_nom,
    'prc_ppp_ind':      df_ppp,
    'ei_bsco_m':        df_cci,
    'demo_pjan':        df_pop,
    'earn_nt_net':      df_earn,
    'avia_tf_aca':      df_avia,
    'avia_paoc':        df_pax,
    'avia_tf_apal':     df_apal,
}

rows = []
for name, df in all_datasets.items():
    if df is not None:
        tc = 'TIME_PERIOD' if 'TIME_PERIOD' in df.columns else None
        rows.append({
            'Dataset': name,
            'Rows': f'{len(df):,}',
            'Cols': df.shape[1],
            'Frequency': df['freq'].iloc[0] if 'freq' in df.columns else '—',
            'Start': df[tc].min() if tc else '—',
            'End': df[tc].max() if tc else '—',
            'Countries': df['geo'].nunique() if 'geo' in df.columns else '—',
            'Null OBS_VALUE': f"{df['OBS_VALUE'].isna().sum():,}" if 'OBS_VALUE' in df.columns else '—',
        })
    else:
        rows.append({'Dataset': name, 'Rows': 'MISSING'})

df_inventory = pd.DataFrame(rows)
print('\n' + '=' * 100)
print('  DATA INVENTORY — All Raw Datasets')
print('=' * 100)
print(df_inventory.to_string(index=False))


  DATA INVENTORY — All Raw Datasets
         Dataset   Rows  Cols Frequency   Start     End Countries Null OBS_VALUE
    tour_occ_nim  9,601    11   Monthly 2003-01 2025-11         2              0
    tour_cap_nat  1,885    11    Annual    1990    2024         3              0
    tour_lfsq6r2 12,401    13 Quarterly 2008-Q1 2025-Q3         2          1,535
   prc_hicp_midx  4,067    10   Monthly 2003-01 2025-12        15              0
   prc_hicp_aind  1,476    10    Annual    2003    2025        15              1
   ert_bil_eur_m  1,668    10   Monthly 2003-01 2026-02         —              0
namq_10_gdp_real  1,459    11 Quarterly 1995-Q1 2025-Q4        12              0
 namq_10_gdp_nom    246    11 Quarterly 1995-Q1 2025-Q3         1              0
     prc_ppp_ind  1,056    10    Annual    1995    2024        36              0
       ei_bsco_m  6,411    11   Monthly 2005-01 2026-02        26              0
       demo_pjan  1,629    11    Annual    1990    2025        52       

### 5.2 Filter Reference Table

The table below documents the **exact filter values** discovered in this notebook. All subsequent notebooks (01–08) should use these values when subsetting the raw data.

| Dataset | Filter Column | Value to Use | Purpose |
|---------|--------------|-------------|--------|
| `tour_occ_nim` | `geo` | `'Poland'` | Destination country |
| `tour_occ_nim` | `c_resid` | `'Foreign country'` | Foreign tourists (aggregate, no country breakdown) |
| `tour_occ_nim` | `unit` | `'Number'` | Absolute overnight stays |
| `tour_occ_nim` | `nace_r2` | `'Hotels; holiday and other short-stay accommodation; camping grounds, recreational vehicle parks and trailer parks'` | All accommodation types |
| `prc_hicp_midx` | `coicop` | `'All-items HICP'` | General price level (pre-filtered) |
| `prc_hicp_midx` | `unit` | `'Index, 2015=100'` | Index base year (pre-filtered) |
| `ert_bil_eur_m` | `currency` | Full names: `'Polish zloty'`, `'Pound sterling'`, `'Swedish krona'`, `'Czech koruna'`, `'Hungarian forint'` | Non-Eurozone exchange rates |
| `ert_bil_eur_m` | `statinfo` | `'Average'` | Monthly average rate (pre-filtered) |
| `namq_10_gdp_real` | `na_item` | `'Gross domestic product at market prices'` | GDP measure (pre-filtered) |
| `namq_10_gdp_real` | `unit` | `'Chain linked volumes (2010), million euro'` | Real GDP in constant prices |
| `namq_10_gdp_real` | `s_adj` | `'Seasonally and calendar adjusted data'`<sup>[8](#fn8)</sup> | Adjusted series |
| `ei_bsco_m` | `indic` | `'Consumer confidence indicator'` | Sentiment measure (pre-filtered) |
| `ei_bsco_m` | `s_adj` | `'Seasonally adjusted data, not calendar adjusted data'` | Adjusted series |
| `avia_tf_aca` | `tra_meas` | `'Passengers seats available'` | Air connectivity (seats) |
| `avia_tf_aca` | `tra_cov` | `'Total transport'` | All transport types (pre-filtered) |
| `demo_pjan` | `age` | `'Total'` | Total population (pre-filtered) |
| `demo_pjan` | `sex` | `'Total'` | Both sexes (pre-filtered) |

### 5.3 Known Data Gaps and Limitations

The following issues were identified during inspection and must be addressed in the feature engineering stage (Notebook 04):

1. **UK GDP ends at 2020-Q3.** The UK stopped reporting to Eurostat after Brexit. The gap (2020-Q4 onwards) will be filled using IMF WEO or OECD quarterly GDP data.

2. **UK Consumer Confidence is missing entirely** from the Eurostat `ei_bsco_m` dataset. An alternative source (GfK UK Consumer Confidence Index) will be required.

3. **UK HICP ends earlier** than other countries (~215 periods vs. 276 for Eurozone members). Post-Brexit CPI from ONS (UK national statistics) can serve as a supplement.

4. **Croatia HICP coverage starts late** (only available from 2023 in this extract for some categories). Croatia joined the EU in 2013 and the Eurozone in 2023, so early data may need to be sourced from national statistics.

5. **GDP is quarterly** — requires interpolation to monthly frequency (cubic spline, following standard practice).

6. **Population and net earnings are annual** — used for descriptive context and normalisation only, not as monthly model inputs.

7. **`tour_occ_nim` does not contain origin-country breakdowns** — `c_resid` distinguishes only Domestic / Foreign / Total. Origin-country effects enter through exogenous variables, not through the dependent variable.

8. **Air passengers (`avia_paoc`) are endogenous** — used for descriptive validation only, never as a predictor.

---

## Conclusion

All **16 raw datasets** have been loaded and structurally verified. The key findings are:

- The target variable (`tour_occ_nim`, foreign overnight stays in Poland) is available monthly from 2003 to 2025-11 with no missing values. The effective study period starts in 2011 (when reliable monthly data begins for all required exogenous variables).
- All required origin-country exogenous variables (HICP, GDP, exchange rates, consumer confidence) are present with adequate temporal coverage, except for the UK, which requires supplementary sources post-Brexit.
- Aviation data covers 13 Polish airports with three transport measures (seats, flights, passengers) from 2010 onwards.
- The Filter Reference Table (Section 5.2) provides the exact column values for subsetting each dataset in all downstream notebooks.

**Next step:** Proceed to [Notebook 01 — Exploratory Data Analysis & Descriptive Statistics](./01_eda_descriptive_statistics.ipynb).

---



## Glossary

<a id="fn1"></a><sup>1</sup> **Cubic spline interpolation:** A piecewise polynomial method used to estimate intermediate data points between known values. In this project, it converts quarterly GDP into a smooth monthly series, preserving the original quarterly totals while avoiding the artificial jumps that simpler methods (e.g. step interpolation) would produce.

<a id="fn2"></a><sup>2</sup> **HICP (Harmonised Index of Consumer Prices):** The standardised consumer price index published by Eurostat for all EU/EEA member states. Unlike national CPIs, HICP uses a common methodology across countries, enabling direct cross-country price comparisons. Base year: 2015 = 100.

<a id="fn3"></a><sup>3</sup> **Chain-linked volumes (CLV):** A method for computing real (inflation-adjusted) GDP by chaining together year-on-year volume changes. Unlike fixed-base-year deflation, chain-linking updates the price weights annually, reducing substitution bias in long time series.

<a id="fn4"></a><sup>4</sup> **IATA slot allocation:** The system governed by the International Air Transport Association (IATA) through which airports allocate take-off and landing rights to airlines, typically decided 6–12 months in advance. This lag makes airline seat capacity a *predetermined* variable relative to short-term tourism demand fluctuations.

<a id="fn5"></a><sup>5</sup> **Endogeneity:** A situation in which an explanatory variable is correlated with the error term of the model, violating a key assumption of causal inference. In our context, air passenger volumes are endogenous because higher tourism demand directly causes more passengers — making them unsuitable as an independent predictor without correction (e.g. instrumental variables or lagging).

<a id="fn6"></a><sup>6</sup> **LCC (Low-Cost Carrier):** Airlines operating with a no-frills, cost-minimisation business model (e.g. Ryanair, Wizz Air). Their market entry is considered a *supply-side shock* — driven by route profitability analysis and slot availability rather than by the tourism demand we are modelling — making it a genuinely exogenous event.

<a id="fn7"></a><sup>7</sup> **COICOP (Classification of Individual Consumption According to Purpose):** The UN classification system used by Eurostat to categorise consumer expenditure items. `CP00` (All-items) captures the general price level; sub-categories like `CP11` (Hotels, cafés, restaurants) capture sector-specific inflation.

<a id="fn8"></a><sup>8</sup> **Seasonal and calendar adjustment:** Statistical procedures that remove recurring seasonal patterns (e.g. summer peaks) and calendar effects (e.g. number of working days, leap years) from time series data. This isolates the underlying trend and cyclical components, making the series more suitable for economic analysis.

<a id="fn9"></a><sup>9</sup> **NACE (Nomenclature of Economic Activities):** The European standard classification of productive economic activities. In tourism contexts, relevant codes include I-55 (Accommodation), I-56 (Food and beverage services), and H-51 (Air transport).

<a id="fn10"></a><sup>10</sup> **PPP (Purchasing Power Parity):** A metric that compares price levels across countries by measuring how much a basket of goods and services costs in each country relative to a reference (here, EU27 2020 = 100). A PPP index of 60 means prices are 40% lower than the EU average.

---
*Notebook 00 complete. All data verified and documented.*